In [1]:

###################################################################
########## IMPORTS DES LIBRAIRIES - FORMATS - CHEMINS... ##########
###################################################################

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import json
from datetime import datetime
from dateutil.relativedelta import relativedelta


In [2]:
import os
import pandas as pd
# import geopandas as gpd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# 1. Chargement du fichier .env pour récupérer tes identifiants secrets
load_dotenv()

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_name = "retail_db"

# 2. DETECTION DE L'ENVIRONNEMENT (Le correctif pour ton erreur)
# Si on est dans Docker, un dossier '/opt/airflow' existe. Sinon, on est sur ton Windows.
if os.path.exists("/opt/airflow"):
    db_host = "postgres-gis"  # Connexion interne Docker
    print("[Info] Exécution détectée dans le conteneur DOCKER.")
else:
    db_host = "localhost"     # Connexion depuis ton VS Code Windows
    print("[Info] Exécution détectée en LOCAL (Windows VS Code).")

# 3. Construction de la chaîne de connexion
DATABASE_URL = f"postgresql://{db_user}:{db_password}@{db_host}:5432/{db_name}"
engine = create_engine(DATABASE_URL)

# 4. Test de la connexion
with engine.connect() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis;"))
    conn.commit()

print("[OK] Connexion sécurisée établie avec succès ! extension PostGIS active.")


python-dotenv could not parse statement starting at line 20
python-dotenv could not parse statement starting at line 21


[Info] Exécution détectée en LOCAL (Windows VS Code).
[OK] Connexion sécurisée établie avec succès ! extension PostGIS active.


#### table logement

In [5]:
import pandas as pd
from sqlalchemy import text

query_logement = text("""
    SELECT *
    FROM fin_caracteristiques_logement
    ORDER BY code_commune, annee_recensement
""")

with engine.connect() as conn:
    df_logement = pd.read_sql(query_logement, conn)

print(df_logement.shape)
print(df_logement.head())
print(df_logement["annee_recensement"].unique())

(2532, 17)
  code_zone_insee  annee_recensement  logements_10_19_ans  logements_1_piece  \
0  2026-COM-75056               2017         234759.08383       263138.04296   
1  2026-COM-75056               2023         198821.63183       261901.41711   
2  2026-COM-77001               2017            148.58681            8.54964   
3  2026-COM-77001               2023            140.32916            1.06465   
4  2026-COM-77002               2017             78.56030            8.94991   

   logements_20_29_ans  logements_2_4_ans  logements_2_pieces  \
0         117450.31250       267134.39926        360437.41744   
1         138351.28940       271313.38030        348906.93966   
2             67.95784           55.83149            11.66424   
3             82.42404           66.53873             5.32427   
4             42.76067           53.69945            19.88868   

   logements_30_ans_ou_plus  logements_3_pieces  logements_4_pieces  \
0              154795.67821        267711.7434

In [6]:
print("Dimensions :", df_logement.shape)

print("\nColonnes :")
print(df_logement.columns.tolist())

print("\nAnnées :")
print(sorted(df_logement["annee_recensement"].dropna().unique()))

print("\nNombre de communes par année :")
print(
    df_logement
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)

print("\nAperçu :")
display(df_logement.head())

Dimensions : (2532, 17)

Colonnes :
['code_zone_insee', 'annee_recensement', 'logements_10_19_ans', 'logements_1_piece', 'logements_20_29_ans', 'logements_2_4_ans', 'logements_2_pieces', 'logements_30_ans_ou_plus', 'logements_3_pieces', 'logements_4_pieces', 'logements_5_9_ans', 'logements_5_pieces_ou_plus', 'logements_moins_2_ans', 'code_commune', 'millesime_referentiel_geo', 'niveau_geo', 'nom_commune']

Années :
[np.int64(2017), np.int64(2023)]

Nombre de communes par année :
annee_recensement
2017    1266
2023    1266
Name: code_commune, dtype: int64

Aperçu :


,code_zone_insee,annee_recensement,logements_10_19_ans,logements_1_piece,logements_20_29_ans,logements_2_4_ans,logements_2_pieces,logements_30_ans_ou_plus,logements_3_pieces,logements_4_pieces,logements_5_9_ans,logements_5_pieces_ou_plus,logements_moins_2_ans,code_commune,millesime_referentiel_geo,niveau_geo,nom_commune
0,2026-COM-75056,2017,234759.08383,263138.04296,117450.31250,267134.39926,360437.41744,154795.67821,267711.74341,146924.55996,191711.69375,103411.34700,175771.94322,75056,2026,COM,Paris
1,2026-COM-75056,2023,198821.63183,261901.41711,138351.28940,271313.38030,348906.93966,148080.43683,265404.16931,144776.45608,196659.40251,103533.37426,171296.21555,75056,2026,COM,Paris
2,2026-COM-77001,2017,148.58681,8.54964,67.95784,55.83149,11.66424,84.56348,37.07821,109.52308,70.85014,295.18484,34.21023,77001,2026,COM,Achères-la-Forêt
3,2026-COM-77001,2023,140.32916,1.06465,82.42404,66.53873,5.32427,105.40909,36.25013,109.31837,69.35022,348.66740,36.57357,77001,2026,COM,Achères-la-Forêt
4,2026-COM-77002,2017,78.56030,8.94991,42.76067,53.69945,19.88868,58.67162,38.78293,86.51577,54.69388,167.06494,32.81633,77002,2026,COM,Amillis


In [7]:
# Copie de travail
df_logement = df_logement.copy()

# On garde uniquement les informations utiles au modèle
colonnes_logement = [
    "code_commune",
    "nom_commune",
    "annee_recensement",
    "logements_10_19_ans",
    "logements_1_piece",
    "logements_20_29_ans",
    "logements_2_4_ans",
    "logements_2_pieces",
    "logements_30_ans_ou_plus",
    "logements_3_pieces",
    "logements_4_pieces",
    "logements_5_9_ans",
    "logements_5_pieces_ou_plus",
    "logements_moins_2_ans"
]

df_logement = df_logement[colonnes_logement].copy()

# Vérification de l'unicité commune × année
controle = (
    df_logement
    .groupby(["code_commune", "annee_recensement"])
    .size()
    .reset_index(name="nb_lignes")
)

print("Doublons commune × année :")
print(controle[controle["nb_lignes"] > 1])

print("\nDimensions :", df_logement.shape)
print("\nAnnées :", sorted(df_logement["annee_recensement"].unique()))

Doublons commune × année :
Empty DataFrame
Columns: [code_commune, annee_recensement, nb_lignes]
Index: []

Dimensions : (2532, 14)

Années : [np.int64(2017), np.int64(2023)]


In [ ]:
df_logement_wide = (
    df_logement
    .pivot(
        index=["code_commune", "nom_commune"],
        columns="annee_recensement",
        values=[
            "logements_10_19_ans",
            "logements_1_piece",
            "logements_20_29_ans",
            "logements_2_4_ans",
            "logements_2_pieces",
            "logements_30_ans_ou_plus",
            "logements_3_pieces",
            "logements_4_pieces",
            "logements_5_9_ans",
            "logements_5_pieces_ou_plus",
            "logements_moins_2_ans"
        ]
    )
    .reset_index()
)

# Aplatir les noms de colonnes
df_logement_wide.columns = [
    "_".join(str(x) for x in col if str(x) != "")
    if isinstance(col, tuple)
    else col
    for col in df_logement_wide.columns
]

print(df_logement_wide.shape)
print(df_logement_wide.columns.tolist())
df_logement_wide

(1266, 24)
['code_commune', 'nom_commune', 'logements_10_19_ans_2017', 'logements_10_19_ans_2023', 'logements_1_piece_2017', 'logements_1_piece_2023', 'logements_20_29_ans_2017', 'logements_20_29_ans_2023', 'logements_2_4_ans_2017', 'logements_2_4_ans_2023', 'logements_2_pieces_2017', 'logements_2_pieces_2023', 'logements_30_ans_ou_plus_2017', 'logements_30_ans_ou_plus_2023', 'logements_3_pieces_2017', 'logements_3_pieces_2023', 'logements_4_pieces_2017', 'logements_4_pieces_2023', 'logements_5_9_ans_2017', 'logements_5_9_ans_2023', 'logements_5_pieces_ou_plus_2017', 'logements_5_pieces_ou_plus_2023', 'logements_moins_2_ans_2017', 'logements_moins_2_ans_2023']


#### table emploi forme juridique

In [9]:
import pandas as pd
from sqlalchemy import text

query_emploi_forme = text("""
    SELECT *
    FROM fin_emploi_forme_juridique
    ORDER BY code_commune, annee_recensement
""")

with engine.connect() as conn:
    df_emploi_forme = pd.read_sql(query_emploi_forme, conn)

print("Dimensions :", df_emploi_forme.shape)

print("\nColonnes :")
print(df_emploi_forme.columns.tolist())

print("\nAnnées :")
print(sorted(df_emploi_forme["annee_recensement"].dropna().unique()))

print("\nNombre de communes par année :")
print(
    df_emploi_forme
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)

print("\nAperçu :")
display(df_emploi_forme.head())

Dimensions : (1266, 12)

Colonnes :
['code_zone_insee', 'annee_recensement', 'emplois_entreprises_individuelles', 'emplois_secteur_public_et_prive_melange', 'emplois_entreprises_privees_toutes_formes', 'unites_locales_entreprises_individuelles', 'unites_locales_secteur_public_et_prive_melange', 'unites_locales_entreprises_privees_toutes_formes', 'code_commune', 'millesime_referentiel_geo', 'niveau_geo', 'nom_commune']

Années :
[np.int64(2024)]

Nombre de communes par année :
annee_recensement
2024    1266
Name: code_commune, dtype: int64

Aperçu :


,code_zone_insee,annee_recensement,emplois_entreprises_individuelles,emplois_secteur_public_et_prive_melange,emplois_entreprises_privees_toutes_formes,unites_locales_entreprises_individuelles,unites_locales_secteur_public_et_prive_melange,unites_locales_entreprises_privees_toutes_formes,code_commune,millesime_referentiel_geo,niveau_geo,nom_commune
0,2026-COM-75056,2024,358360.0,1285561.0,927201.0,3169.0,114327.0,111158.0,75056,2026,COM,Paris
1,2026-COM-77001,2024,35.0,129.0,94.0,3.0,14.0,11.0,77001,2026,COM,Achères-la-Forêt
2,2026-COM-77002,2024,21.0,182.0,161.0,3.0,25.0,22.0,77002,2026,COM,Amillis
3,2026-COM-77003,2024,8.0,34.0,26.0,3.0,6.0,3.0,77003,2026,COM,Amponville
4,2026-COM-77004,2024,4.0,13.0,9.0,2.0,7.0,5.0,77004,2026,COM,Andrezel


In [10]:
df_emploi_forme = df_emploi_forme[
    [
        "code_commune",
        "nom_commune",
        "annee_recensement",
        "emplois_entreprises_individuelles",
        "emplois_secteur_public_et_prive_melange",
        "emplois_entreprises_privees_toutes_formes",
        "unites_locales_entreprises_individuelles",
        "unites_locales_secteur_public_et_prive_melange",
        "unites_locales_entreprises_privees_toutes_formes"
    ]
].copy()

In [11]:
variables_emploi = [
    "emplois_entreprises_individuelles",
    "emplois_secteur_public_et_prive_melange",
    "emplois_entreprises_privees_toutes_formes",
    "unites_locales_entreprises_individuelles",
    "unites_locales_secteur_public_et_prive_melange",
    "unites_locales_entreprises_privees_toutes_formes"
]

df_emploi_forme = df_emploi_forme.rename(
    columns={
        variable: f"{variable}_2024"
        for variable in variables_emploi
    }
)

# On peut retirer la colonne année puisque l'année est maintenant
# intégrée directement dans les noms de variables
df_emploi_forme = df_emploi_forme.drop(columns=["annee_recensement"])

In [12]:
print("Dimensions :", df_emploi_forme.shape)

print("\nNombre de communes :", df_emploi_forme["code_commune"].nunique())

print("\nDoublons sur code_commune :")
print(df_emploi_forme["code_commune"].duplicated().sum())

print("\nValeurs manquantes :")
print(df_emploi_forme.isna().sum())

display(df_emploi_forme.head())

Dimensions : (1266, 8)

Nombre de communes : 1266

Doublons sur code_commune :
0

Valeurs manquantes :
code_commune                                             0
nom_commune                                              0
emplois_entreprises_individuelles_2024                   0
emplois_secteur_public_et_prive_melange_2024             0
emplois_entreprises_privees_toutes_formes_2024           0
unites_locales_entreprises_individuelles_2024            0
unites_locales_secteur_public_et_prive_melange_2024      0
unites_locales_entreprises_privees_toutes_formes_2024    0
dtype: int64


,code_commune,nom_commune,emplois_entreprises_individuelles_2024,emplois_secteur_public_et_prive_melange_2024,emplois_entreprises_privees_toutes_formes_2024,unites_locales_entreprises_individuelles_2024,unites_locales_secteur_public_et_prive_melange_2024,unites_locales_entreprises_privees_toutes_formes_2024
0,75056,Paris,358360.0,1285561.0,927201.0,3169.0,114327.0,111158.0
1,77001,Achères-la-Forêt,35.0,129.0,94.0,3.0,14.0,11.0
2,77002,Amillis,21.0,182.0,161.0,3.0,25.0,22.0
3,77003,Amponville,8.0,34.0,26.0,3.0,6.0,3.0
4,77004,Andrezel,4.0,13.0,9.0,2.0,7.0,5.0


#### table population emploi

In [13]:
import pandas as pd
from sqlalchemy import text

query_population_emploi = text("""
    SELECT *
    FROM fin_population_emploi
    ORDER BY code_commune, annee_recensement
""")

with engine.connect() as conn:
    df_population_emploi = pd.read_sql(
        query_population_emploi,
        conn
    )

print("Dimensions :", df_population_emploi.shape)

print("\nColonnes :")
print(df_population_emploi.columns.tolist())

print("\nAnnées :")
print(
    sorted(
        df_population_emploi["annee_recensement"]
        .dropna()
        .unique()
    )
)

print("\nNombre de communes par année :")
print(
    df_population_emploi
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)

print("\nAperçu :")
display(df_population_emploi.head())

Dimensions : (1266, 6)

Colonnes :
['code_zone_insee', 'annee_recensement', 'nombre_salaries', 'nombre_etablissements', 'code_commune', 'nom_commune']

Années :
[np.int64(2024)]

Nombre de communes par année :
annee_recensement
2024    1266
Name: code_commune, dtype: int64

Aperçu :


,code_zone_insee,annee_recensement,nombre_salaries,nombre_etablissements,code_commune,nom_commune
0,2026-COM-75056,2024,2010517.0,188002.0,75056,Paris
1,2026-COM-77001,2024,158.0,35.0,77001,Achères-la-Forêt
2,2026-COM-77002,2024,220.0,39.0,77002,Amillis
3,2026-COM-77003,2024,42.0,12.0,77003,Amponville
4,2026-COM-77004,2024,19.0,13.0,77004,Andrezel


In [14]:
print("\nDoublons commune × année :")

controle = (
    df_population_emploi
    .groupby(["code_commune", "annee_recensement"])
    .size()
    .reset_index(name="nb_lignes")
)

display(controle[controle["nb_lignes"] > 1])


Doublons commune × année :


,code_commune,annee_recensement,nb_lignes


In [15]:
df_population_emploi = df_population_emploi[
    [
        "code_commune",
        "nom_commune",
        "annee_recensement",
        "nombre_salaries",
        "nombre_etablissements"
    ]
].copy()

In [16]:
df_population_emploi = df_population_emploi.rename(
    columns={
        "nombre_salaries": "nombre_salaries_2024",
        "nombre_etablissements": "nombre_etablissements_2024"
    }
)

df_population_emploi = df_population_emploi.drop(
    columns=["annee_recensement"]
)

In [17]:
print("Dimensions :", df_population_emploi.shape)

print(
    "Nombre de communes :",
    df_population_emploi["code_commune"].nunique()
)

print(
    "Doublons code_commune :",
    df_population_emploi["code_commune"].duplicated().sum()
)

print("\nValeurs manquantes :")
print(df_population_emploi.isna().sum())

display(df_population_emploi.head())

Dimensions : (1266, 4)
Nombre de communes : 1266
Doublons code_commune : 0

Valeurs manquantes :
code_commune                  0
nom_commune                   0
nombre_salaries_2024          0
nombre_etablissements_2024    0
dtype: int64


,code_commune,nom_commune,nombre_salaries_2024,nombre_etablissements_2024
0,75056,Paris,2010517.0,188002.0
1,77001,Achères-la-Forêt,158.0,35.0
2,77002,Amillis,220.0,39.0
3,77003,Amponville,42.0,12.0
4,77004,Andrezel,19.0,13.0


#### population pouvoir d'achat 

In [18]:
import pandas as pd
from sqlalchemy import text

query_pouvoir_achat = text("""
    SELECT *
    FROM fin_pouvoir_achat_population
    ORDER BY code_commune, annee_recensement
""")

with engine.connect() as conn:
    df_pouvoir_achat = pd.read_sql(
        query_pouvoir_achat,
        conn
    )

print("Dimensions :", df_pouvoir_achat.shape)

print("\nColonnes :")
print(df_pouvoir_achat.columns.tolist())

print("\nAnnées :")
print(
    sorted(
        df_pouvoir_achat["annee_recensement"]
        .dropna()
        .unique()
    )
)

print("\nNombre de communes par année :")
print(
    df_pouvoir_achat
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)

print("\nAperçu :")
display(df_pouvoir_achat.head())

Dimensions : (1252, 6)

Colonnes :
['code_zone_insee', 'annee_recensement', 'niveau_vie_median', 'taux_pauvrete_60', 'code_commune', 'nom_commune']

Années :
[np.int64(2023)]

Nombre de communes par année :
annee_recensement
2023    1252
Name: code_commune, dtype: int64

Aperçu :


,code_zone_insee,annee_recensement,niveau_vie_median,taux_pauvrete_60,code_commune,nom_commune
0,2026-COM-75056,2023,33650.0,16.8,75056,Paris
1,2026-COM-77001,2023,34840.0,NaN,77001,Achères-la-Forêt
2,2026-COM-77002,2023,29130.0,NaN,77002,Amillis
3,2026-COM-77003,2023,30310.0,NaN,77003,Amponville
4,2026-COM-77004,2023,31970.0,NaN,77004,Andrezel


In [19]:
controle = (
    df_pouvoir_achat
    .groupby(["code_commune", "annee_recensement"])
    .size()
    .reset_index(name="nb_lignes")
)

print("Doublons commune × année :")
display(controle[controle["nb_lignes"] > 1])

Doublons commune × année :


,code_commune,annee_recensement,nb_lignes


In [20]:
df_pouvoir_achat = df_pouvoir_achat.rename(
    columns={
        "niveau_vie_median": "niveau_vie_median_2023",
        "taux_pauvrete_60": "taux_pauvrete_60_2023"
    }
)

df_pouvoir_achat = df_pouvoir_achat.drop(
    columns=["annee_recensement"]
)

In [21]:
print("Dimensions :", df_pouvoir_achat.shape)

print(
    "Nombre de communes :",
    df_pouvoir_achat["code_commune"].nunique()
)

print(
    "Doublons code_commune :",
    df_pouvoir_achat["code_commune"].duplicated().sum()
)

print("\nValeurs manquantes :")
print(df_pouvoir_achat.isna().sum())

display(df_pouvoir_achat.head())

Dimensions : (1252, 5)
Nombre de communes : 1252
Doublons code_commune : 0

Valeurs manquantes :
code_zone_insee             0
niveau_vie_median_2023      0
taux_pauvrete_60_2023     709
code_commune                0
nom_commune                 0
dtype: int64


,code_zone_insee,niveau_vie_median_2023,taux_pauvrete_60_2023,code_commune,nom_commune
0,2026-COM-75056,33650.0,16.8,75056,Paris
1,2026-COM-77001,34840.0,NaN,77001,Achères-la-Forêt
2,2026-COM-77002,29130.0,NaN,77002,Amillis
3,2026-COM-77003,30310.0,NaN,77003,Amponville
4,2026-COM-77004,31970.0,NaN,77004,Andrezel


#### table mobilité domicile travail

In [22]:
import pandas as pd
from sqlalchemy import text

query_mobilite = text("""
    SELECT *
    FROM fin_mobilite_domicile_travail
    ORDER BY code_commune, annee_recensement
""")

with engine.connect() as conn:
    df_mobilite = pd.read_sql(
        query_mobilite,
        conn
    )

print("Dimensions :", df_mobilite.shape)

print("\nColonnes :")
print(df_mobilite.columns.tolist())

print("\nAnnées :")
print(
    sorted(
        df_mobilite["annee_recensement"]
        .dropna()
        .unique()
    )
)

print("\nNombre de communes par année :")
print(
    df_mobilite
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)

print("\nAperçu :")
display(df_mobilite.head())

Dimensions : (2532, 12)

Colonnes :
['code_zone_insee', 'annee_recensement', 'travaille_dans_la_commune', 'travaille_dans_le_departement', 'travaille_dans_un_autre_departement_de_la_region', 'travaille_hors_commune', 'travaille_hors_france_ou_non_precise', 'travaille_hors_region', 'code_commune', 'millesime_referentiel_geo', 'niveau_geo', 'nom_commune']

Années :
[np.int64(2017), np.int64(2023)]

Nombre de communes par année :
annee_recensement
2017    1266
2023    1266
Name: code_commune, dtype: int64

Aperçu :


,code_zone_insee,annee_recensement,travaille_dans_la_commune,travaille_dans_le_departement,travaille_dans_un_autre_departement_de_la_region,travaille_hors_commune,travaille_hors_france_ou_non_precise,travaille_hors_region,code_commune,millesime_referentiel_geo,niveau_geo,nom_commune
0,2026-COM-75056,2017,737170.63648,0.00000,322635.19055,342413.27286,5615.23373,14162.84858,75056,None,None,Paris
1,2026-COM-75056,2023,715780.62511,0.00000,332899.34276,355116.01547,5035.32121,17181.35150,75056,None,None,Paris
2,2026-COM-77001,2017,120.10614,212.32440,211.78010,445.53123,1.96017,19.46655,77001,None,None,Achères-la-Forêt
3,2026-COM-77001,2023,88.69753,235.62367,229.65689,495.33179,0.00000,30.05123,77001,None,None,Achères-la-Forêt
4,2026-COM-77002,2017,70.60483,205.84788,79.55473,292.36365,2.98330,3.97774,77002,None,None,Amillis


In [23]:
controle = (
    df_mobilite
    .groupby(["code_commune", "annee_recensement"])
    .size()
    .reset_index(name="nb_lignes")
)

print("Doublons commune × année :")
display(controle[controle["nb_lignes"] > 1])

Doublons commune × année :


,code_commune,annee_recensement,nb_lignes


In [24]:
print(
    df_mobilite
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)

annee_recensement
2017    1266
2023    1266
Name: code_commune, dtype: int64


In [25]:
df_mobilite = df_mobilite[
    [
        "code_commune",
        "nom_commune",
        "annee_recensement",
        "travaille_dans_la_commune",
        "travaille_dans_le_departement",
        "travaille_dans_un_autre_departement_de_la_region",
        "travaille_hors_commune",
        "travaille_hors_france_ou_non_precise",
        "travaille_hors_region"
    ]
].copy()

In [26]:
df_mobilite_wide = (
    df_mobilite
    .pivot(
        index=["code_commune", "nom_commune"],
        columns="annee_recensement",
        values=[
            "travaille_dans_la_commune",
            "travaille_dans_le_departement",
            "travaille_dans_un_autre_departement_de_la_region",
            "travaille_hors_commune",
            "travaille_hors_france_ou_non_precise",
            "travaille_hors_region"
        ]
    )
    .reset_index()
)

In [27]:
df_mobilite_wide.columns = [
    "_".join(str(x) for x in col if str(x) != "")
    if isinstance(col, tuple)
    else col
    for col in df_mobilite_wide.columns
]

df_mobilite_wide.columns.name = None

In [28]:
print("Dimensions :", df_mobilite_wide.shape)

print(
    "Nombre de communes :",
    df_mobilite_wide["code_commune"].nunique()
)

print(
    "Doublons code_commune :",
    df_mobilite_wide["code_commune"].duplicated().sum()
)

print("\nValeurs manquantes :")
print(df_mobilite_wide.isna().sum())

display(df_mobilite_wide.head())

Dimensions : (1266, 14)
Nombre de communes : 1266
Doublons code_commune : 0

Valeurs manquantes :
code_commune                                             0
nom_commune                                              0
travaille_dans_la_commune_2017                           0
travaille_dans_la_commune_2023                           0
travaille_dans_le_departement_2017                       0
travaille_dans_le_departement_2023                       0
travaille_dans_un_autre_departement_de_la_region_2017    0
travaille_dans_un_autre_departement_de_la_region_2023    0
travaille_hors_commune_2017                              0
travaille_hors_commune_2023                              0
travaille_hors_france_ou_non_precise_2017                0
travaille_hors_france_ou_non_precise_2023                0
travaille_hors_region_2017                               0
travaille_hors_region_2023                               0
dtype: int64


,code_commune,nom_commune,travaille_dans_la_commune_2017,travaille_dans_la_commune_2023,travaille_dans_le_departement_2017,travaille_dans_le_departement_2023,travaille_dans_un_autre_departement_de_la_region_2017,travaille_dans_un_autre_departement_de_la_region_2023,travaille_hors_commune_2017,travaille_hors_commune_2023,travaille_hors_france_ou_non_precise_2017,travaille_hors_france_ou_non_precise_2023,travaille_hors_region_2017,travaille_hors_region_2023
0,75056,Paris,737170.63648,715780.62511,0.00000,0.00000,322635.19055,332899.34276,342413.27286,355116.01547,5615.23373,5035.32121,14162.84858,17181.35150
1,77001,Achères-la-Forêt,120.10614,88.69753,212.32440,235.62367,211.78010,229.65689,445.53123,495.33179,1.96017,0.00000,19.46655,30.05123
2,77002,Amillis,70.60483,64.81709,205.84788,248.64191,79.55473,52.41261,292.36365,313.85822,2.98330,1.00021,3.97774,11.80349
3,77003,Amponville,38.02454,19.04237,82.97273,84.56358,65.89790,55.68256,164.93000,155.18845,1.03590,1.01446,15.02347,13.92785
4,77004,Andrezel,26.86439,22.82017,83.68920,88.04094,30.88927,45.59180,116.57629,134.63569,0.00000,0.00000,1.99782,1.00294


In [29]:
import pandas as pd
from sqlalchemy import text

query_reference_population = text("""
    SELECT *
    FROM fin_reference_population
    ORDER BY code_commune, annee_recensement
""")

with engine.connect() as conn:
    df_reference_population = pd.read_sql(
        query_reference_population,
        conn
    )

print("Dimensions :", df_reference_population.shape)

print("\nColonnes :")
print(df_reference_population.columns.tolist())

print("\nAnnées :")
print(
    sorted(
        df_reference_population["annee_recensement"]
        .dropna()
        .unique()
    )
)

print("\nNombre de communes par année :")
print(
    df_reference_population
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)

print("\nAperçu :")
display(df_reference_population.head())

Dimensions : (1266, 7)

Colonnes :
['code_zone_insee', 'annee_recensement', 'population_au_sens_large_non_residente', 'population_municipale', 'population_totale', 'code_commune', 'nom_commune']

Années :
[np.int64(2023)]

Nombre de communes par année :
annee_recensement
2023    1266
Name: code_commune, dtype: int64

Aperçu :


,code_zone_insee,annee_recensement,population_au_sens_large_non_residente,population_municipale,population_totale,code_commune,nom_commune
0,2025-COM-75056,2023,15634.0,2103778.0,2119412.0,75056,Paris
1,2025-COM-77001,2023,32.0,1191.0,1223.0,77001,Achères-la-Forêt
2,2025-COM-77002,2023,5.0,830.0,835.0,77002,Amillis
3,2025-COM-77003,2023,8.0,349.0,357.0,77003,Amponville
4,2025-COM-77004,2023,2.0,336.0,338.0,77004,Andrezel


In [30]:
controle = (
    df_reference_population
    .groupby(["code_commune", "annee_recensement"])
    .size()
    .reset_index(name="nb_lignes")
)

print("Doublons commune × année :")
display(controle[controle["nb_lignes"] > 1])

Doublons commune × année :


,code_commune,annee_recensement,nb_lignes


In [31]:
df_reference_population = df_reference_population[
    [
        "code_commune",
        "nom_commune",
        "annee_recensement",
        "population_au_sens_large_non_residente",
        "population_municipale",
        "population_totale"
    ]
].copy()

In [32]:
df_reference_population = df_reference_population.rename(
    columns={
        "population_au_sens_large_non_residente":
            "population_au_sens_large_non_residente_2023",
        "population_municipale":
            "population_municipale_2023",
        "population_totale":
            "population_totale_2023"
    }
)

df_reference_population = df_reference_population.drop(
    columns=["annee_recensement"]
)

In [33]:
print("Dimensions :", df_reference_population.shape)

print(
    "Nombre de communes :",
    df_reference_population["code_commune"].nunique()
)

print(
    "Doublons code_commune :",
    df_reference_population["code_commune"].duplicated().sum()
)

print("\nValeurs manquantes :")
print(df_reference_population.isna().sum())

display(df_reference_population.head())

Dimensions : (1266, 5)
Nombre de communes : 1266
Doublons code_commune : 0

Valeurs manquantes :
code_commune                                   0
nom_commune                                    0
population_au_sens_large_non_residente_2023    0
population_municipale_2023                     0
population_totale_2023                         0
dtype: int64


,code_commune,nom_commune,population_au_sens_large_non_residente_2023,population_municipale_2023,population_totale_2023
0,75056,Paris,15634.0,2103778.0,2119412.0
1,77001,Achères-la-Forêt,32.0,1191.0,1223.0
2,77002,Amillis,5.0,830.0,835.0
3,77003,Amponville,8.0,349.0,357.0
4,77004,Andrezel,2.0,336.0,338.0


In [34]:
import pandas as pd
from sqlalchemy import text

query_recensement_population = text("""
    SELECT *
    FROM fin_recensement_population
    ORDER BY code_commune, annee_recensement
""")

with engine.connect() as conn:
    df_recensement_population = pd.read_sql(
        query_recensement_population,
        conn
    )

print("Dimensions :", df_recensement_population.shape)

print("\nColonnes :")
print(df_recensement_population.columns.tolist())

print("\nAnnées :")
print(
    sorted(
        df_recensement_population["annee_recensement"]
        .dropna()
        .unique()
    )
)

print("\nNombre de communes par année :")
print(
    df_recensement_population
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)

print("\nAperçu :")
display(df_recensement_population.head())

Dimensions : (2532, 18)

Colonnes :
['code_zone_insee', 'annee_recensement', 'population_femmes_15_24_ans', 'population_femmes_20_64_ans', 'population_femmes_25_39_ans', 'population_femmes_40_54_ans', 'population_femmes_55_64_ans', 'population_femmes_65_ans_et_plus', 'population_femmes_moins_de_15_ans', 'population_hommes_15_24_ans', 'population_hommes_20_64_ans', 'population_hommes_25_39_ans', 'population_hommes_40_54_ans', 'population_hommes_55_64_ans', 'population_hommes_65_ans_et_plus', 'population_hommes_moins_de_15_ans', 'code_commune', 'nom_commune']

Années :
[np.int64(2017), np.int64(2023)]

Nombre de communes par année :
annee_recensement
2017    1266
2023    1266
Name: code_commune, dtype: int64

Aperçu :


,code_zone_insee,annee_recensement,population_femmes_15_24_ans,population_femmes_20_64_ans,population_femmes_25_39_ans,population_femmes_40_54_ans,population_femmes_55_64_ans,population_femmes_65_ans_et_plus,population_femmes_moins_de_15_ans,population_hommes_15_24_ans,population_hommes_20_64_ans,population_hommes_25_39_ans,population_hommes_40_54_ans,population_hommes_55_64_ans,population_hommes_65_ans_et_plus,population_hommes_moins_de_15_ans,code_commune,nom_commune
0,2026-COM-75056,2017,156628.07670,734578.03119,289922.25801,217959.61753,129078.58331,215781.20416,149252.95972,133226.81558,670144.02184,273590.47006,207517.92455,111164.92183,149643.98645,153759.18212,75056,Paris
1,2026-COM-75056,2023,161961.91248,705247.08321,279018.79281,199372.77055,124520.14053,218245.61597,132320.39648,133867.97450,644314.31776,261920.93496,190681.33483,112165.74548,153763.56465,135938.81676,75056,Paris
2,2026-COM-77001,2017,60.27395,344.40513,74.40235,173.42868,72.23879,92.19940,98.19962,82.05203,339.36545,58.83414,169.23923,68.15643,82.85583,93.11954,77001,Achères-la-Forêt
3,2026-COM-77001,2023,58.62215,361.47290,84.17933,158.49481,98.48529,125.40563,92.15548,56.53079,338.67106,72.12427,140.98488,107.34178,107.41838,89.25721,77001,Achères-la-Forêt
4,2026-COM-77002,2017,37.78850,235.77024,81.54360,75.66634,65.63266,86.97368,70.60483,34.80520,230.88741,71.59926,75.66634,69.69973,75.46514,79.55473,77002,Amillis


In [ ]:
# supprimer lescolonnes inutiles
# df_recensement_population = df_recensement_population.drop(columns={'population_femmes_20_64_ans', 'population_hommes_20_64_ans'})

In [42]:
controle = (
    df_recensement_population
    .groupby(["code_commune", "annee_recensement"])
    .size()
    .reset_index(name="nb_lignes")
)

print("Doublons commune × année :")
display(controle[controle["nb_lignes"] > 1])

Doublons commune × année :


,code_commune,annee_recensement,nb_lignes


In [43]:
print("\nNombre de communes par année :")

print(
    df_recensement_population
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)


Nombre de communes par année :
annee_recensement
2017    1266
2023    1266
Name: code_commune, dtype: int64


In [44]:
variables_recensement = [
    "population_femmes_15_24_ans",
    "population_femmes_25_39_ans",
    "population_femmes_40_54_ans",
    "population_femmes_55_64_ans",
    "population_femmes_65_ans_et_plus",
    "population_femmes_moins_de_15_ans",
    "population_hommes_15_24_ans",
    "population_hommes_25_39_ans",
    "population_hommes_40_54_ans",
    "population_hommes_55_64_ans",
    "population_hommes_65_ans_et_plus",
    "population_hommes_moins_de_15_ans"
]

df_recensement_population = df_recensement_population[
    [
        "code_commune",
        "nom_commune",
        "annee_recensement"
    ] + variables_recensement
].copy()

In [45]:
df_recensement_population_wide = (
    df_recensement_population
    .pivot(
        index=["code_commune", "nom_commune"],
        columns="annee_recensement",
        values=variables_recensement
    )
    .reset_index()
)

In [46]:
df_recensement_population_wide.columns = [
    "_".join(str(x) for x in col if str(x) != "")
    if isinstance(col, tuple)
    else col
    for col in df_recensement_population_wide.columns
]

df_recensement_population_wide.columns.name = None

In [47]:
print("Dimensions :", df_recensement_population_wide.shape)

print(
    "Nombre de communes :",
    df_recensement_population_wide["code_commune"].nunique()
)

print(
    "Doublons code_commune :",
    df_recensement_population_wide["code_commune"].duplicated().sum()
)

print("\nValeurs manquantes :")
print(
    df_recensement_population_wide
    .isna()
    .sum()
)

display(df_recensement_population_wide.head())

Dimensions : (1266, 26)
Nombre de communes : 1266
Doublons code_commune : 0

Valeurs manquantes :
code_commune                              0
nom_commune                               0
population_femmes_15_24_ans_2017          0
population_femmes_15_24_ans_2023          0
population_femmes_25_39_ans_2017          0
population_femmes_25_39_ans_2023          0
population_femmes_40_54_ans_2017          0
population_femmes_40_54_ans_2023          0
population_femmes_55_64_ans_2017          0
population_femmes_55_64_ans_2023          0
population_femmes_65_ans_et_plus_2017     0
population_femmes_65_ans_et_plus_2023     0
population_femmes_moins_de_15_ans_2017    0
population_femmes_moins_de_15_ans_2023    0
population_hommes_15_24_ans_2017          0
population_hommes_15_24_ans_2023          0
population_hommes_25_39_ans_2017          0
population_hommes_25_39_ans_2023          0
population_hommes_40_54_ans_2017          0
population_hommes_40_54_ans_2023          0
population_hommes_55_6

,code_commune,nom_commune,population_femmes_15_24_ans_2017,population_femmes_15_24_ans_2023,population_femmes_25_39_ans_2017,population_femmes_25_39_ans_2023,population_femmes_40_54_ans_2017,population_femmes_40_54_ans_2023,population_femmes_55_64_ans_2017,population_femmes_55_64_ans_2023,...,population_hommes_25_39_ans_2017,population_hommes_25_39_ans_2023,population_hommes_40_54_ans_2017,population_hommes_40_54_ans_2023,population_hommes_55_64_ans_2017,population_hommes_55_64_ans_2023,population_hommes_65_ans_et_plus_2017,population_hommes_65_ans_et_plus_2023,population_hommes_moins_de_15_ans_2017,population_hommes_moins_de_15_ans_2023
0,75056,Paris,156628.07670,161961.91248,289922.25801,279018.79281,217959.61753,199372.77055,129078.58331,124520.14053,...,273590.47006,261920.93496,207517.92455,190681.33483,111164.92183,112165.74548,149643.98645,153763.56465,153759.18212,135938.81676
1,77001,Achères-la-Forêt,60.27395,58.62215,74.40235,84.17933,173.42868,158.49481,72.23879,98.48529,...,58.83414,72.12427,169.23923,140.98488,68.15643,107.34178,82.85583,107.41838,93.11954,89.25721
2,77002,Amillis,37.78850,43.53257,81.54360,67.94482,75.66634,72.47546,65.63266,68.93133,...,71.59926,67.26280,75.66634,80.72952,69.69973,66.67313,75.46514,79.21900,79.55473,66.97936
3,77003,Amponville,15.64689,14.82746,27.77116,33.71298,46.73371,38.77358,29.93859,27.09351,...,26.96049,25.83806,54.97619,40.63548,24.64506,29.14371,30.01859,33.11695,23.07851,34.32118
4,77004,Andrezel,10.88164,10.43978,24.88112,40.54600,25.86928,25.94447,35.06913,19.36490,...,20.91479,31.15641,35.82185,33.26156,32.02432,24.62594,20.05207,24.47625,26.70996,43.21788


#### table commerces idf

In [67]:
import pandas as pd
from sqlalchemy import text

query_infrastructures = text("""
    SELECT *
    FROM fin_infrastructures_de_proximite
    ORDER BY code_commune, annee_recensement
""")

with engine.connect() as conn:
    df_infrastructures = pd.read_sql(
        query_infrastructures,
        conn
    )

print("Dimensions :", df_infrastructures.shape)

print("\nColonnes :")
print(df_infrastructures.columns.tolist())

print("\nAnnées :")
print(
    sorted(
        df_infrastructures["annee_recensement"]
        .dropna()
        .unique()
    )
)

print("\nNombre de communes par année :")
print(
    df_infrastructures
    .groupby("annee_recensement")["code_commune"]
    .nunique()
)

print("\nAperçu :")
display(df_infrastructures.head())

Dimensions : (1190, 35)

Colonnes :
['code_zone_insee', 'annee_recensement', 'banque', 'bureau_de_poste', 'restauration_rapide', 'hypermarche_grand_magasin', 'supermarche', 'superette', 'epicerie', 'boulangerie_patisserie', 'magasin_vetements', 'magasin_chaussures', 'magasin_meubles', 'parfumerie_cosmetique', 'electromenager_audio_video_informatique', 'librairie', 'papeterie_presse', 'ecole_maternelle', 'ecole_primaire', 'ecole_elementaire', 'college', 'lycee', 'urgences', 'pharmacie', 'gare_nationale', 'gare_regionale', 'transport_ferroviaire_local_rer_metro_transilien', 'salle_non_specialisee_sport', 'bowling', 'salle_remise_en_forme', 'gymnase_multisports', 'code_commune', 'millesime_referentiel_geo', 'niveau_geo', 'nom_commune']

Années :
[np.int64(2025)]

Nombre de communes par année :
annee_recensement
2025    1190
Name: code_commune, dtype: int64

Aperçu :


,code_zone_insee,annee_recensement,banque,bureau_de_poste,restauration_rapide,hypermarche_grand_magasin,supermarche,superette,epicerie,boulangerie_patisserie,...,gare_regionale,transport_ferroviaire_local_rer_metro_transilien,salle_non_specialisee_sport,bowling,salle_remise_en_forme,gymnase_multisports,code_commune,millesime_referentiel_geo,niveau_geo,nom_commune
0,2026-COM-75056,2025,1265.0,128.0,20911.0,44.0,715.0,368.0,2106.0,2307.0,...,27.0,NaN,88.0,6.0,244.0,267.0,75056,2026,COM,Paris
1,2026-COM-77001,2025,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,77001,2026,COM,Achères-la-Forêt
2,2026-COM-77002,2025,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,77002,2026,COM,Amillis
3,2026-COM-77003,2025,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,77003,2026,COM,Amponville
4,2026-COM-77004,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,77004,2026,COM,Andrezel


In [69]:
# Vérification de l'unicité commune × année
controle = (
    df_infrastructures
    .groupby(["code_commune", "annee_recensement"])
    .size()
    .reset_index(name="nb_lignes")
)

print("Doublons commune × année :")
display(controle[controle["nb_lignes"] > 1])


# Toutes les variables d'infrastructure
variables_infrastructures = [
    "banque",
    "bureau_de_poste",
    "restauration_rapide",
    "hypermarche_grand_magasin",
    "supermarche",
    "superette",
    "epicerie",
    "boulangerie_patisserie",
    "magasin_vetements",
    "magasin_chaussures",
    "magasin_meubles",
    "parfumerie_cosmetique",
    "electromenager_audio_video_informatique",
    "librairie",
    "papeterie_presse",
    "ecole_maternelle",
    "ecole_primaire",
    "ecole_elementaire",
    "college",
    "lycee",
    "urgences",
    "pharmacie",
    "gare_nationale",
    "gare_regionale",
    "transport_ferroviaire_local_rer_metro_transilien",
    "salle_non_specialisee_sport",
    "bowling",
    "salle_remise_en_forme",
    "gymnase_multisports"
]


# Sélection des colonnes utiles
df_infrastructures_2025 = df_infrastructures[
    [
        "code_commune",
        "nom_commune",
        "annee_recensement"
    ] + variables_infrastructures
].copy()


# Ajouter l'année aux variables
df_infrastructures_2025 = df_infrastructures_2025.rename(
    columns={
        variable: f"{variable}_2025"
        for variable in variables_infrastructures
    }
)


# L'année est désormais intégrée dans les noms de variables
df_infrastructures_2025 = df_infrastructures_2025.drop(
    columns=["annee_recensement"]
)



Doublons commune × année :


,code_commune,annee_recensement,nb_lignes


In [70]:
print("Dimensions :", df_infrastructures_2025.shape)

print(
    "Nombre de communes :",
    df_infrastructures_2025["code_commune"].nunique()
)

print(
    "Doublons code_commune :",
    df_infrastructures_2025["code_commune"].duplicated().sum()
)

print("\nValeurs manquantes :")
test = df_infrastructures_2025.isna().sum()
print(test)

print("\nTypes :")
print(df_infrastructures_2025.dtypes)

display(df_infrastructures_2025.head())

Dimensions : (1190, 31)
Nombre de communes : 1190
Doublons code_commune : 0

Valeurs manquantes :
code_commune                                                0
nom_commune                                                 0
banque_2025                                               805
bureau_de_poste_2025                                      746
restauration_rapide_2025                                  258
hypermarche_grand_magasin_2025                           1018
supermarche_2025                                          730
superette_2025                                            918
epicerie_2025                                             589
boulangerie_patisserie_2025                               475
magasin_vetements_2025                                    755
magasin_chaussures_2025                                   998
magasin_meubles_2025                                      867
parfumerie_cosmetique_2025                                875
electromenager_audio_video_informa

,code_commune,nom_commune,banque_2025,bureau_de_poste_2025,restauration_rapide_2025,hypermarche_grand_magasin_2025,supermarche_2025,superette_2025,epicerie_2025,boulangerie_patisserie_2025,...,lycee_2025,urgences_2025,pharmacie_2025,gare_nationale_2025,gare_regionale_2025,transport_ferroviaire_local_rer_metro_transilien_2025,salle_non_specialisee_sport_2025,bowling_2025,salle_remise_en_forme_2025,gymnase_multisports_2025
0,75056,Paris,1265.0,128.0,20911.0,44.0,715.0,368.0,2106.0,2307.0,...,173.0,16.0,864.0,7.0,27.0,NaN,88.0,6.0,244.0,267.0
1,77001,Achères-la-Forêt,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
2,77002,Amillis,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
3,77003,Amponville,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,77004,Andrezel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN


In [71]:
# Remplacer les NaN par 0 uniquement dans les variables d'infrastructures 2025

colonnes_infrastructures_2025 = [
    f"{variable}_2025"
    for variable in variables_infrastructures
]

df_infrastructures_2025[colonnes_infrastructures_2025] = (
    df_infrastructures_2025[colonnes_infrastructures_2025]
    .fillna(0)
)

In [72]:
print(
    "Nombre de valeurs manquantes dans les infrastructures :",
    df_infrastructures_2025[colonnes_infrastructures_2025]
    .isna()
    .sum()
    .sum()
)

Nombre de valeurs manquantes dans les infrastructures : 0


In [73]:
display(
    df_infrastructures_2025[
        ["code_commune", "nom_commune"] + colonnes_infrastructures_2025
    ].head(20)
)

,code_commune,nom_commune,banque_2025,bureau_de_poste_2025,restauration_rapide_2025,hypermarche_grand_magasin_2025,supermarche_2025,superette_2025,epicerie_2025,boulangerie_patisserie_2025,...,lycee_2025,urgences_2025,pharmacie_2025,gare_nationale_2025,gare_regionale_2025,transport_ferroviaire_local_rer_metro_transilien_2025,salle_non_specialisee_sport_2025,bowling_2025,salle_remise_en_forme_2025,gymnase_multisports_2025
0,75056,Paris,1265.0,128.0,20911.0,44.0,715.0,368.0,2106.0,2307.0,...,173.0,16.0,864.0,7.0,27.0,0.0,88.0,6.0,244.0,267.0
1,77001,Achères-la-Forêt,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,77002,Amillis,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,77003,Amponville,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,77004,Andrezel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,77005,Annet-sur-Marne,0.0,1.0,6.0,0.0,0.0,0.0,2.0,3.0,...,0.0,0.0,1.0,0.0,0.0,0.0,3.0,0.0,1.0,1.0
6,77006,Arbonne-la-Forêt,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,77007,Argentières,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
8,77008,Armentières-en-Brie,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,77010,Aubepierre-Ozouer-le-Repos,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [74]:
print(
    df_infrastructures_2025[colonnes_infrastructures_2025]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

banque_2025                                              0
bureau_de_poste_2025                                     0
restauration_rapide_2025                                 0
hypermarche_grand_magasin_2025                           0
supermarche_2025                                         0
superette_2025                                           0
epicerie_2025                                            0
boulangerie_patisserie_2025                              0
magasin_vetements_2025                                   0
magasin_chaussures_2025                                  0
magasin_meubles_2025                                     0
parfumerie_cosmetique_2025                               0
electromenager_audio_video_informatique_2025             0
librairie_2025                                           0
papeterie_presse_2025                                    0
ecole_maternelle_2025                                    0
ecole_primaire_2025                                     

#### table commerces idf

In [57]:
# from sqlalchemy import text
# import pandas as pd

# query_commerces = text("""
#     SELECT *
#     FROM fin_localisation_commerces_idf
#     ORDER BY code_postal, annee_recensement
# """)

# with engine.connect() as conn:
#     df_commerces = pd.read_sql(query_commerces, conn)

# print("Dimensions :", df_commerces.shape)
# print("\nColonnes :")
# print(df_commerces.columns.tolist())

# print("\nAnnées disponibles :")
# print(df_commerces["annee_recensement"].unique())

# df_commerces = df_commerces.rename(columns={"code_postal": "code_commune"})
# print("\nNombre de communes :")
# print(df_commerces["code_commune"].nunique())

# print("\nAperçu :")
# display(df_commerces.head())

In [59]:
query_communes = text("""
    SELECT
        code_postal as code_commune,
        nom_commune
    FROM fin_communes_geom
    ORDER BY code_commune
""")

with engine.connect() as conn:
    df_base = pd.read_sql(query_communes, conn)

print("Dimensions :", df_base.shape)
print("Nombre de communes :", df_base["code_commune"].nunique())
print("Doublons :", df_base["code_commune"].duplicated().sum())

display(df_base.head())

Dimensions : (1266, 2)
Nombre de communes : 1266
Doublons : 0


,code_commune,nom_commune
0,75056,Paris
1,77001,Achères-la-Forêt
2,77002,Amillis
3,77003,Amponville
4,77004,Andrezel


In [75]:
dataframes_a_controler = {
    "base": df_base,
    "logement": df_logement_wide,
    "emploi_forme": df_emploi_forme,
    "population_emploi": df_population_emploi,
    "pouvoir_achat": df_pouvoir_achat,
    "mobilite": df_mobilite_wide,
    "reference_population": df_reference_population,
    "recensement_population": df_recensement_population_wide,
    "infrastructures": df_infrastructures_2025
}

for nom, df in dataframes_a_controler.items():
    print(
        f"{nom:25} | "
        f"lignes = {len(df):4} | "
        f"communes = {df['code_commune'].nunique():4} | "
        f"doublons = {df['code_commune'].duplicated().sum():4}"
    )

base                      | lignes = 1266 | communes = 1266 | doublons =    0
logement                  | lignes = 1266 | communes = 1266 | doublons =    0
emploi_forme              | lignes = 1266 | communes = 1266 | doublons =    0
population_emploi         | lignes = 1266 | communes = 1266 | doublons =    0
pouvoir_achat             | lignes = 1252 | communes = 1252 | doublons =    0
mobilite                  | lignes = 1266 | communes = 1266 | doublons =    0
reference_population      | lignes = 1266 | communes = 1266 | doublons =    0
recensement_population    | lignes = 1266 | communes = 1266 | doublons =    0
infrastructures           | lignes = 1190 | communes = 1190 | doublons =    0


In [76]:
for nom, df in dataframes_a_controler.items():
    print(
        f"{nom:25} -> "
        f"type = {df['code_commune'].dtype}"
    )

base                      -> type = str
logement                  -> type = str
emploi_forme              -> type = str
population_emploi         -> type = str
pouvoir_achat             -> type = str
mobilite                  -> type = str
reference_population      -> type = str
recensement_population    -> type = str
infrastructures           -> type = str


In [77]:
df_modele = df_base.merge(
    df_logement_wide,
    on=["code_commune", "nom_commune"],
    how="left",
    validate="one_to_one"
)

print("Dimensions :", df_modele.shape)
print("Communes :", df_modele["code_commune"].nunique())
print("Doublons :", df_modele["code_commune"].duplicated().sum())

Dimensions : (1266, 24)
Communes : 1266
Doublons : 0


In [78]:
# ============================================================
# CONSTRUCTION DE LA TABLE EXPLICATIVE
# 1 ligne = 1 commune
# ============================================================

df_modele = df_base.copy()

print("Départ :", df_modele.shape)


# ------------------------------------------------------------
# 1. Caractéristiques des logements
# 2017 / 2023
# ------------------------------------------------------------

df_modele = df_modele.merge(
    df_logement_wide.drop(columns=["nom_commune"], errors="ignore"),
    on="code_commune",
    how="left",
    validate="one_to_one"
)

print("Après logement :", df_modele.shape)


# ------------------------------------------------------------
# 2. Mobilité domicile-travail
# 2017 / 2023
# ------------------------------------------------------------

df_modele = df_modele.merge(
    df_mobilite_wide.drop(columns=["nom_commune"], errors="ignore"),
    on="code_commune",
    how="left",
    validate="one_to_one"
)

print("Après mobilité :", df_modele.shape)


# ------------------------------------------------------------
# 3. Recensement de la population
# 2017 / 2023
# ------------------------------------------------------------

df_modele = df_modele.merge(
    df_recensement_population_wide.drop(
        columns=["nom_commune"],
        errors="ignore"
    ),
    on="code_commune",
    how="left",
    validate="one_to_one"
)

print("Après recensement population :", df_modele.shape)


# ------------------------------------------------------------
# 4. Référence population
# 2023
# ------------------------------------------------------------

df_modele = df_modele.merge(
    df_reference_population.drop(
        columns=["nom_commune"],
        errors="ignore"
    ),
    on="code_commune",
    how="left",
    validate="one_to_one"
)

print("Après référence population :", df_modele.shape)


# ------------------------------------------------------------
# 5. Pouvoir d'achat
# 2023
# ------------------------------------------------------------

df_modele = df_modele.merge(
    df_pouvoir_achat.drop(
        columns=["nom_commune"],
        errors="ignore"
    ),
    on="code_commune",
    how="left",
    validate="one_to_one"
)

print("Après pouvoir d'achat :", df_modele.shape)


# ------------------------------------------------------------
# 6. Emploi et forme juridique
# 2024
# ------------------------------------------------------------

df_modele = df_modele.merge(
    df_emploi_forme.drop(
        columns=["nom_commune"],
        errors="ignore"
    ),
    on="code_commune",
    how="left",
    validate="one_to_one"
)

print("Après emploi / forme juridique :", df_modele.shape)


# ------------------------------------------------------------
# 7. Population et établissements
# 2024
# ------------------------------------------------------------

df_modele = df_modele.merge(
    df_population_emploi.drop(
        columns=["nom_commune"],
        errors="ignore"
    ),
    on="code_commune",
    how="left",
    validate="one_to_one"
)

print("Après population / établissements :", df_modele.shape)


# ------------------------------------------------------------
# 8. Infrastructures de proximité
# 2025
# ------------------------------------------------------------

df_modele = df_modele.merge(
    df_infrastructures_2025.drop(
        columns=["nom_commune"],
        errors="ignore"
    ),
    on="code_commune",
    how="left",
    validate="one_to_one"
)

print("Après infrastructures :", df_modele.shape)

Départ : (1266, 2)
Après logement : (1266, 24)
Après mobilité : (1266, 36)
Après recensement population : (1266, 60)
Après référence population : (1266, 63)
Après pouvoir d'achat : (1266, 66)
Après emploi / forme juridique : (1266, 72)
Après population / établissements : (1266, 74)
Après infrastructures : (1266, 103)


In [79]:
controle_na = (
    df_modele
    .isna()
    .sum()
    .reset_index()
)

controle_na.columns = ["variable", "nb_valeurs_manquantes"]

controle_na["pct_manquant"] = (
    controle_na["nb_valeurs_manquantes"]
    / len(df_modele)
    * 100
)

display(
    controle_na
    .sort_values("pct_manquant", ascending=False)
)

,variable,nb_valeurs_manquantes,pct_manquant
65,taux_pauvrete_60_2023,723,57.109005
101,salle_remise_en_forme_2025,76,6.003160
100,bowling_2025,76,6.003160
102,gymnase_multisports_2025,76,6.003160
89,ecole_maternelle_2025,76,6.003160
...,...,...,...
43,population_femmes_55_64_ans_2023,0,0.000000
39,population_femmes_25_39_ans_2023,0,0.000000
71,unites_locales_entreprises_privees_toutes_form...,0,0.000000
72,nombre_salaries_2024,0,0.000000


In [80]:
# Communes avec taux de pauvreté manquant
communes_pauvrete_na = df_modele[
    df_modele["taux_pauvrete_60_2023"].isna()
][
    ["code_commune", "nom_commune", "niveau_vie_median_2023"]
].copy()

print("Nombre de communes avec taux de pauvreté manquant :",
      len(communes_pauvrete_na))

display(communes_pauvrete_na.head(30))

Nombre de communes avec taux de pauvreté manquant : 723


,code_commune,nom_commune,niveau_vie_median_2023
1,77001,Achères-la-Forêt,34840.0
2,77002,Amillis,29130.0
3,77003,Amponville,30310.0
4,77004,Andrezel,31970.0
6,77006,Arbonne-la-Forêt,34140.0
7,77007,Argentières,32250.0
8,77008,Armentières-en-Brie,26210.0
9,77009,Arville,27070.0
10,77010,Aubepierre-Ozouer-le-Repos,29690.0
11,77011,Aufferville,28210.0


In [81]:
print(
    "Parmi les 723 communes :"
)

print(
    "niveau_vie_median_2023 également manquant :",
    communes_pauvrete_na["niveau_vie_median_2023"].isna().sum()
)

Parmi les 723 communes :
niveau_vie_median_2023 également manquant : 14


In [82]:
display(
    df_modele.loc[
        df_modele["taux_pauvrete_60_2023"].isna(),
        [
            "code_commune",
            "nom_commune",
            "population_municipale_2023",
            "population_totale_2023",
            "nombre_etablissements_2024"
        ]
    ].describe(include="all")
)

,code_commune,nom_commune,population_municipale_2023,population_totale_2023,nombre_etablissements_2024
count,723,723,723.000000,723.000000,723.000000
unique,723,720,NaN,NaN,NaN
top,77001,Blandy,NaN,NaN,NaN
freq,1,2,NaN,NaN,NaN
mean,NaN,NaN,739.218534,752.686030,20.533887
std,NaN,NaN,493.813497,503.665816,18.773911
min,NaN,NaN,32.000000,34.000000,2.000000
25%,NaN,NaN,344.500000,349.000000,9.000000
50%,NaN,NaN,628.000000,642.000000,15.000000
75%,NaN,NaN,1007.000000,1024.000000,25.000000


In [83]:
colonnes_infrastructures_2025 = [
    f"{variable}_2025"
    for variable in variables_infrastructures
]

communes_infra_na = df_modele[
    df_modele[colonnes_infrastructures_2025]
    .isna()
    .all(axis=1)
][
    ["code_commune", "nom_commune"]
].copy()

print(
    "Communes avec toutes les infrastructures manquantes :",
    len(communes_infra_na)
)

display(communes_infra_na.head(30))

Communes avec toutes les infrastructures manquantes : 76


,code_commune,nom_commune
9,77009,Arville
13,77013,Aulnoy
15,77015,Baby
27,77029,Beauvoir
34,77036,Boisdon
36,77038,Boissettes
54,77056,Burcy
57,77059,Bussy-Saint-Martin
86,77090,La Chapelle-Saint-Sulpice
93,77098,Châteaubleau


In [84]:
print(df_modele.dtypes.value_counts())

display(
    pd.DataFrame({
        "variable": df_modele.columns,
        "type": df_modele.dtypes.astype(str).values,
        "nb_uniques": [
            df_modele[col].nunique(dropna=True)
            for col in df_modele.columns
        ],
        "nb_na": [
            df_modele[col].isna().sum()
            for col in df_modele.columns
        ]
    })
)

float64    100
str          3
Name: count, dtype: int64


,variable,type,nb_uniques,nb_na
0,code_commune,str,1266,0
1,nom_commune,str,1262,0
2,logements_10_19_ans_2017,float64,1205,0
3,logements_10_19_ans_2023,float64,1200,0
4,logements_1_piece_2017,float64,947,0
...,...,...,...,...
98,transport_ferroviaire_local_rer_metro_transili...,float64,3,76
99,salle_non_specialisee_sport_2025,float64,20,76
100,bowling_2025,float64,4,76
101,salle_remise_en_forme_2025,float64,16,76


In [86]:
df_modele = df_modele.drop(columns=['code_zone_insee'])

In [93]:
df_modele.to_excel("table_modelisation.xlsx", index=False)

In [87]:
df_numerique = df_modele.select_dtypes(include="number")

print("Nombre de variables numériques :", df_numerique.shape[1])

display(
    df_numerique.describe().T
)

Nombre de variables numériques : 100


,count,mean,std,min,25%,50%,75%,max
logements_10_19_ans_2017,1266.0,897.114072,6727.065856,0.0,51.695820,128.045015,602.524923,234759.08383
logements_10_19_ans_2023,1266.0,839.032156,5734.619950,1.0,47.000000,115.911335,537.667458,198821.63183
logements_1_piece_2017,1266.0,479.922334,7430.144943,0.0,1.034667,8.730600,112.083407,263138.04296
logements_1_piece_2023,1266.0,517.181483,7405.745085,0.0,1.037200,8.683590,115.567870,261901.41711
logements_20_29_ans_2017,1266.0,453.043738,3361.798615,0.0,30.042135,73.620300,327.921300,117450.31250
...,...,...,...,...,...,...,...,...
transport_ferroviaire_local_rer_metro_transilien_2025,1190.0,0.027731,0.169313,0.0,0.000000,0.000000,0.000000,2.00000
salle_non_specialisee_sport_2025,1190.0,1.147059,3.286344,0.0,0.000000,1.000000,1.000000,88.00000
bowling_2025,1190.0,0.033613,0.247197,0.0,0.000000,0.000000,0.000000,6.00000
salle_remise_en_forme_2025,1190.0,1.026891,7.295006,0.0,0.000000,0.000000,1.000000,244.00000


#### Variables constantes

In [88]:
# Variables avec une seule valeur non manquante
variables_constantes = []

for col in df_modele.columns:
    if df_modele[col].nunique(dropna=True) <= 1:
        variables_constantes.append(col)

print("Variables constantes ou sans valeur :")
print(variables_constantes)

Variables constantes ou sans valeur :
[]


#### variables peu variables

In [90]:
# Nombre de valeurs distinctes par variable
nb_uniques = (
    df_modele
    .nunique(dropna=True)
    .sort_values()
)

display(nb_uniques.to_frame("nb_valeurs_uniques"))

,nb_valeurs_uniques
transport_ferroviaire_local_rer_metro_transilien_2025,3
gare_nationale_2025,3
urgences_2025,4
bowling_2025,4
hypermarche_grand_magasin_2025,6
...,...
travaille_hors_commune_2017,1239
logements_5_pieces_ou_plus_2023,1246
travaille_hors_commune_2023,1248
nom_commune,1262


#### Statistiques descriptives

In [91]:
df_numerique = df_modele.select_dtypes(include="number")

stats_variables = df_numerique.describe().T

stats_variables["nb_na"] = df_numerique.isna().sum()

stats_variables["pct_na"] = (
    df_numerique.isna().mean() * 100
)

display(stats_variables)

,count,mean,std,min,25%,50%,75%,max,nb_na,pct_na
logements_10_19_ans_2017,1266.0,897.114072,6727.065856,0.0,51.695820,128.045015,602.524923,234759.08383,0,0.00000
logements_10_19_ans_2023,1266.0,839.032156,5734.619950,1.0,47.000000,115.911335,537.667458,198821.63183,0,0.00000
logements_1_piece_2017,1266.0,479.922334,7430.144943,0.0,1.034667,8.730600,112.083407,263138.04296,0,0.00000
logements_1_piece_2023,1266.0,517.181483,7405.745085,0.0,1.037200,8.683590,115.567870,261901.41711,0,0.00000
logements_20_29_ans_2017,1266.0,453.043738,3361.798615,0.0,30.042135,73.620300,327.921300,117450.31250,0,0.00000
...,...,...,...,...,...,...,...,...,...,...
transport_ferroviaire_local_rer_metro_transilien_2025,1190.0,0.027731,0.169313,0.0,0.000000,0.000000,0.000000,2.00000,76,6.00316
salle_non_specialisee_sport_2025,1190.0,1.147059,3.286344,0.0,0.000000,1.000000,1.000000,88.00000,76,6.00316
bowling_2025,1190.0,0.033613,0.247197,0.0,0.000000,0.000000,0.000000,6.00000,76,6.00316
salle_remise_en_forme_2025,1190.0,1.026891,7.295006,0.0,0.000000,0.000000,1.000000,244.00000,76,6.00316
